# 01 · Train the player / ball / hoop detector

One YOLO detector for every on-court object the pipeline needs: `Player`, `Ref`, `Ball`, `Hoop`
(plus `Clock`, `Scoreboard`, `Overlay`, which the pipeline ignores but which help the model learn
what *isn't* a player).

Changes from the original training notebooks:

* **The dataset downloads in YOLO format** (`"yolov8"`). The player notebook asked for `"yolov5x-seg.pt"`,
  which is a weights file name, not a format.
* **Base weights exist.** `yolov5.pt` is not a downloadable checkpoint, which is why that run failed.
  We use `yolo11m.pt` (swap in `yolo11l.pt` / `yolo11x.pt` on an A100).
* **Runs save to Google Drive** with `save_period`, so a Colab disconnect (the ball run stopped at
  epoch 244 of 250) resumes instead of starting over.
* **API key from Colab Secrets**, not hard-coded.
* **Per-class validation**, because "all classes mAP" hides a weak `Ball` class, and the ball is the
  hardest and most important object.
* **Optional ball-only model** trained at higher resolution for small, blurred balls.

In [ ]:
# Runtime > Change runtime type > GPU before running.
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

# Everything that must survive a Colab disconnect (runs, weights) lives in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)

In [ ]:
!pip -q install "ultralytics>=8.3" "roboflow>=1.1"

## Dataset

In [ ]:
# Store your Roboflow key in Colab's Secrets panel (key icon on the left) as
# ROBOFLOW_API_KEY. Never paste it into a cell: notebooks get shared and committed.
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("workspace-5ujvu").project("basketball-players-fy4c2-vfsuv")
dataset = project.version(17).download("yolov8", location="/content/datasets/players-v17")
DATA_YAML = f"{dataset.location}/data.yaml"
print(open(DATA_YAML).read())

In [ ]:
# Label counts per split. Your v17 validation split has only 32 images; if a class has
# fewer than ~100 validation boxes its mAP is noise. Re-split in Roboflow (70/20/10) if so.
import collections, glob, yaml
names = yaml.safe_load(open(DATA_YAML))["names"]
for split in ["train", "valid", "test"]:
    counts = collections.Counter()
    files = glob.glob(f"{dataset.location}/{split}/labels/*.txt")
    for f in files:
        for line in open(f):
            counts[names[int(line.split()[0])]] += 1
    print(f"{split:5s} {len(files):5d} images  ", dict(counts))

## Train

Broadcast frames are wide and the ball is ~15-25 px, so train at `imgsz=1280` and keep mosaic on
(it creates many small-object examples). `fliplr` is fine here: left/right does not change a
player's class. `batch=-1` lets Ultralytics pick the largest batch that fits the GPU.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "detector_yolo11m_1280"
last = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/last.pt"
resume = os.path.exists(last)
model = YOLO(last if resume else "yolo11m.pt")
model.train(
    data=DATA_YAML, imgsz=1280, epochs=150, patience=30, batch=-1,
    cos_lr=True, close_mosaic=15, save_period=10,
    project=f"{DRIVE_ROOT}/runs", name=RUN_NAME, exist_ok=True, resume=resume,
)

## Validate per class

In [ ]:
best = YOLO(f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt")
metrics = best.val(data=DATA_YAML, imgsz=1280, split="valid")
for i, name in best.names.items():
    p, r, ap50, ap = metrics.box.class_result(i) if i in metrics.box.ap_class_index else (0, 0, 0, 0)
    print(f"{name:12s} P={p:.3f} R={r:.3f} mAP50={ap50:.3f} mAP50-95={ap:.3f}")

In [ ]:
import shutil
shutil.copy(f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt", f"{DRIVE_ROOT}/models/player_detector.pt")
print("Saved", f"{DRIVE_ROOT}/models/player_detector.pt")

## Optional: dedicated ball detector

If `Ball` recall above is below ~0.85, train a ball-only model. One class means all capacity goes to
the ball, and a larger `imgsz` keeps it from shrinking to a few pixels. Point
`models.ball_detector` in `configs/default.yaml` at the result; the pipeline then uses it for the
ball and the main model for everything else.

In [ ]:
import pathlib, shutil, yaml
src = pathlib.Path(dataset.location)
dst = pathlib.Path("/content/datasets/ball-only")
ball_id = [i for i, n in enumerate(names) if n.lower() == "ball"][0]
for split in ["train", "valid", "test"]:
    if not (src / split).exists():
        continue
    (dst / split / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copytree(src / split / "images", dst / split / "images", dirs_exist_ok=True)
    for f in (src / split / "labels").glob("*.txt"):
        keep = [" ".join(["0"] + l.split()[1:]) for l in open(f) if int(l.split()[0]) == ball_id]
        (dst / split / "labels" / f.name).write_text("\n".join(keep) + ("\n" if keep else ""))
yaml.safe_dump({"path": str(dst), "train": "train/images", "val": "valid/images",
                "test": "test/images", "names": {0: "Ball"}}, open(dst / "data.yaml", "w"))
print(open(dst / "data.yaml").read())

In [ ]:
BALL_RUN = "ball_yolo11s_1536"
last = f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/last.pt"
resume = os.path.exists(last)
ball_model = YOLO(last if resume else "yolo11s.pt")
ball_model.train(data=str(dst / "data.yaml"), imgsz=1536, epochs=200, patience=40, batch=-1,
                 cos_lr=True, close_mosaic=20, save_period=10,
                 project=f"{DRIVE_ROOT}/runs", name=BALL_RUN, exist_ok=True, resume=resume)
shutil.copy(f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/best.pt", f"{DRIVE_ROOT}/models/ball_detector.pt")